In [19]:
import numpy as np
import glob
import os

data_dir = '/Users/muhammad/Desktop/Generated_Embeddings/Tab_ICL/NF-UNSW-NB15-v2/tabicl_embeddings'
 

embedding_files = sorted(glob.glob(os.path.join(data_dir, 'embeddings_batch_*.npy')))
label_files = sorted(glob.glob(os.path.join(data_dir, 'labels_batch_*.npy')))

# Load and stack embeddings
embedding_batches = [np.load(f) for f in embedding_files]  # list of (60000, 512) or (batch, 60000, 512)
# If shape is (31, 60000, 512) per file, flatten each
embedding_batches = [e.reshape(-1, e.shape[-1]) for e in embedding_batches]
x_train = np.concatenate(embedding_batches, axis=0)

# Load and stack labels
label_batches = [np.load(f) for f in label_files]
y_train = np.concatenate([l.reshape(-1) for l in label_batches], axis=0)

In [20]:
x_train.shape


(1860000, 512)

In [21]:
y_train.shape

(1860000,)

In [22]:

unique_labels, counts = np.unique(y_train, return_counts=True)
unique_labels

array(['Analysis', 'Backdoor', 'Benign', 'DoS', 'Exploits', 'Fuzzers',
       'Generic', 'Reconnaissance', 'Shellcode', 'Worms'], dtype='<U14')

In [23]:
binary_arr = np.where(y_train == 'Benign', 0, 1)
unique_labels, counts = np.unique(binary_arr, return_counts=True)
binary_arr.shape


(1860000,)

In [49]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# embeddings = X_train[0]  # Now shape: (N, D)

# label_encoder = LabelEncoder()
# labels_encoded = label_encoder.fit_transform(labels)

# Sanity check
# assert embeddings.shape[0] == labels.shape[0], "Mismatch between embeddings and labels"

# Step 3: Split into train and test sets
# X_train, X_test, y_train, y_test = train_test_split(
#     x_train, binary_arr, test_size=0.2, random_state=42, stratify=binary_arr
# )

split_index = int(len(x_train) * 0.9)  # 80% train, 20% test

X_train = x_train[:split_index]
X_test = x_train[split_index:]

y_train = binary_arr[:split_index]
y_test = binary_arr[split_index:]

# Step 4: Save to .npz
np.savez('custom_data.npz', X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)

# Optional: Verify
print("Saved data.npz with:", np.load('custom_data.npz').files)

Saved data.npz with: ['X_train', 'y_train', 'X_test', 'y_test']


In [50]:
 a= np.load('custom_data.npz')

In [51]:
y_train=a['X_train']
y_train.shape


(1674000, 512)

In [52]:
print(np.unique(a['y_test']))
print(a['y_test'].shape)

[0 1]
(186000,)


In [28]:
unique, counts = np.unique(y_train, return_counts=True)
total = counts.sum()
print(f"Unique labels: {unique}, Counts: {counts}, Total: {total}")


Unique labels: [-5.6739354 -5.672953  -5.6729374 ...  5.2806344  5.281729   5.282678 ], Counts: [1 1 1 ... 2 1 1], Total: 761856000


In [159]:
import numpy as np
import pandas as pd

# Example: assume y_train is stored in a dictionary `a`
y_train = a['y_train']  # Could be a numpy array or pandas Series

# Convert to numpy array for flexibility
y_train = np.array(y_train)

# Count values
unique, counts = np.unique(y_train, return_counts=True)
total = counts.sum()
print(f"Total samples in y_train: {total}")
# Create a DataFrame for better formatting
class_distribution = pd.DataFrame({
    "Label": unique,
    "Count": counts,
    "Percentage": (counts / total) * 100
})

# Map 0/1 to labels (adjust if 0 is attack or benign differently)
label_map = {0: "Benign", 1: "Anomaly"}
class_distribution["Label"] = class_distribution["Label"].map(label_map)

# Display results
print("🔍 Class distribution in y_train:")
print(class_distribution.round(2))


Total samples in y_train: 1488000
🔍 Class distribution in y_train:
     Label    Count  Percentage
0   Benign  1428745       96.02
1  Anomaly    59255        3.98


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Sample dataset: replace this with your own df = pd.read_csv(...)
# Example assumes you have 'Label' as the target column
# df = pd.read_csv('your_dataset.csv')

# 1. Display original dataset info
print("Original Dataset Shape:", df.shape)
print("\nOriginal Class Distribution:")
print(a['Label'].value_counts(normalize=True))

# 2. Cochran's Sample Size Formula
def cochrans_formula(confidence_level=0.95, margin_of_error=0.01, p=0.5):
    """
    Calculate required sample size using Cochran's formula.
    confidence_level: e.g., 0.95 for 95%, 0.99 for 99%
    margin_of_error: e.g., 0.01 for ±1%
    p: estimated proportion (default 0.5 for maximum variance)
    """
    z_scores = {
        0.90: 1.645,
        0.95: 1.96,
        0.99: 2.576
    }
    z = z_scores.get(confidence_level, 1.96)  # Default to 95% if not found
    n = (z**2 * p * (1 - p)) / (margin_of_error**2)
    return int(np.ceil(n))

# Set parameters
confidence_level = 0.99
margin_of_error = 0.01

# Compute required sample size
required_sample_size = cochrans_formula(confidence_level, margin_of_error)

# Ensure we don't sample more than available
max_sample_size = len(df)
if required_sample_size > max_sample_size:
    required_sample_size = max_sample_size

print(f"\nRequired Sample Size (Confidence Level {confidence_level*100}%, Margin of Error ±{margin_of_error*100}%): {required_sample_size}")

# 3. Stratified Sampling from Full Dataset
sampled_df = df.groupby('Label', group_keys=False).apply(
    lambda x: x.sample(frac=(required_sample_size / len(df)), random_state=42)
)

print("\nSampled Dataset Shape:", sampled_df.shape)
print("\nSampled Class Distribution:")
print(sampled_df['Label'].value_counts(normalize=True))

# Optional: Shuffle the sampled dataset
sampled_df = sampled_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Train/Test Split (Stratified)
X = sampled_df.drop('Label', axis=1)
y = sampled_df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,              # 30% for testing
    stratify=y,                 # Preserve class distribution
    random_state=42
)

print("\nTrain set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nTrain set class distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest set class distribution:")
print(y_test.value_counts(normalize=True))

# Optional: Save sampled datasets
sampled_df.to_csv('sampled_dataset.csv', index=False)
X_train.to_csv('train_features.csv', index=False)
X_test.to_csv('test_features.csv', index=False)
y_train.to_csv('train_labels.csv', index=False)
y_test.to_csv('test_labels.csv', index=False)

print("\n✅ Sampling and splitting completed. Files saved.")